# O2 A Plot Bundle Demo

Scientific question: can the Python plotting package produce inspectable, core-backed O2 A validation and diagnostic plots from the prepared reference case?

This notebook displays core-backed charts inline. The tracked validation bundle under `validation/` is not changed.

## Inputs and API Calls

The notebook prepares the native O2 A reference case, collects spectrum, atmospheric-budget, CIA, instrument-response, and radiative-transfer tables, then passes those tables to `zdisamar.plot` for inline display.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "build.zig").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not find repository root")


REPO_ROOT = find_repo_root()
PYTHON_ROOT = REPO_ROOT / "python"
DEFAULT_REFERENCE = REPO_ROOT / "validation" / "data" / "o2a_with_cia_disamar_reference.csv"
MARKERS_NM = (755.0, 760.76, 776.0)
RESIDUAL_THRESHOLD = 1.0e-14


def import_runtime():
    sys.path.insert(0, str(PYTHON_ROOT))
    import altair as alt
    import zdisamar as zd
    import zdisamar.plot as zp

    alt.data_transformers.disable_max_rows()
    return zd, zp


def validation_grid(case) -> np.ndarray:
    return np.linspace(
        case.spectral_grid.start_nm,
        case.spectral_grid.end_nm,
        case.spectral_grid.sample_count,
        dtype=np.float64,
    )


def nearest_grid_values(grid: np.ndarray, targets_nm: tuple[float, ...]) -> np.ndarray:
    indexes = np.unique([int(np.argmin(np.abs(grid - target))) for target in targets_nm])
    return grid[indexes]


def instrument_grid_values(grid: np.ndarray) -> np.ndarray:
    sampled = grid[::35]
    anchors = nearest_grid_values(grid, MARKERS_NM)
    return np.unique(np.concatenate([sampled, anchors]))


def resample_reference(reference: pd.DataFrame, spectrum: pd.DataFrame) -> pd.DataFrame:
    wavelength_nm = spectrum["wavelength_nm"].to_numpy(dtype=float)
    output = {"wavelength_nm": wavelength_nm}
    for quantity in ("reflectance", "radiance", "irradiance"):
        output[quantity] = np.interp(
            wavelength_nm,
            reference["wavelength_nm"].to_numpy(dtype=float),
            reference[quantity].to_numpy(dtype=float),
        )
    return pd.DataFrame(output)


def reflectance_residuals(spectrum: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    residual = spectrum["reflectance"].to_numpy(dtype=float) - reference["reflectance"].to_numpy(
        dtype=float
    )
    return pd.DataFrame(
        {
            "wavelength_nm": spectrum["wavelength_nm"],
            "zig_implementation": spectrum["reflectance"],
            "reference_implementation": reference["reflectance"],
            "residual": residual,
            "abs_residual": np.abs(residual),
            "above_threshold": np.abs(residual) > RESIDUAL_THRESHOLD,
        }
    )


def validation_metrics(spectrum: pd.DataFrame, reference: pd.DataFrame) -> dict[str, object]:
    metrics: dict[str, object] = {
        "sample_count": int(len(spectrum)),
        "wavelength_min_nm": float(spectrum["wavelength_nm"].min()),
        "wavelength_max_nm": float(spectrum["wavelength_nm"].max()),
    }
    for quantity in ("reflectance", "radiance", "irradiance"):
        residual = spectrum[quantity].to_numpy(dtype=float) - reference[quantity].to_numpy(
            dtype=float
        )
        metrics[quantity] = {
            "mae": float(np.mean(np.abs(residual))),
            "rmse": float(np.sqrt(np.mean(np.square(residual)))),
            "max_abs": float(np.max(np.abs(residual))),
            "mean_signed": float(np.mean(residual)),
            "max_abs_wavelength_nm": float(
                spectrum["wavelength_nm"].iloc[int(np.argmax(np.abs(residual)))]
            ),
        }
    return metrics


def progress(message: str) -> None:
    print(f"[python-o2a-plot-bundle] {message}", flush=True)


def collect_core_outputs(zd, zp, library_path: str, reference_path: Path):
    progress("loading native validation case")
    case = zd.o2a_disamar_reference_input(library_path)
    grid = validation_grid(case)
    selected_wavelengths = nearest_grid_values(grid, MARKERS_NM)
    response_wavelengths = instrument_grid_values(grid)

    with zd.prepare(case, library_path=library_path) as prepared:
        progress("running full forward model")
        with prepared.forward_model() as spectrum:
            spectrum_frame = zp.to_dataframe(spectrum)
            reference = resample_reference(pd.read_csv(reference_path), spectrum_frame)

            progress("collecting atmospheric budget")
            with prepared.atmosphere.budget(wavelengths_nm=grid) as budget:
                budget_frame = budget.to_pandas()
            progress("collecting O2-O2 CIA diagnostics")
            with prepared.o2_o2_cia.diagnostics(wavelengths_nm=grid) as cia:
                cia_frame = cia.to_pandas()
            progress("collecting instrument-response diagnostics")
            with prepared.instrument_response.sampling_table(
                wavelengths_nm=response_wavelengths
            ) as response:
                response_frame = response.to_pandas()
            progress("collecting radiative-transfer diagnostics")
            with prepared.radiative_transfer.diagnostics(
                wavelengths_nm=grid, spectrum=spectrum
            ) as rt:
                rt_frame = rt.to_pandas()

    return {
        "case": case,
        "grid": grid,
        "selected_wavelengths": selected_wavelengths,
        "response_wavelengths": response_wavelengths,
        "spectrum": spectrum_frame,
        "reference": reference,
        "residuals": reflectance_residuals(spectrum_frame, reference),
        "metrics": validation_metrics(spectrum_frame, reference),
        "budget": budget_frame,
        "cia": cia_frame,
        "response": response_frame,
        "rt": rt_frame,
    }


def wavelength_slug(wavelength_nm: float) -> str:
    return f"{float(wavelength_nm):08.5f}".replace(".", "_") + "_nm"


def build_plots(zp, bundle: dict[str, object]) -> dict[str, object]:
    spectrum = bundle["spectrum"]
    reference = bundle["reference"]
    selected_wavelengths = [float(value) for value in bundle["selected_wavelengths"]]

    charts: dict[str, object] = {
        "validation/full_sampled_reflectance": zp.validation.residual_highlight_spectrum(
            spectrum,
            reference,
            quantity=zp.fields.REFLECTANCE,
            residual_threshold=RESIDUAL_THRESHOLD,
        ),
        "validation/reflectance_residual_histogram": zp.validation.residual_histogram(
            spectrum,
            reference,
            quantity=zp.fields.REFLECTANCE,
        ),
        "validation/reflectance_residual_by_wavelength": zp.validation.residual_by_wavelength(
            spectrum,
            reference,
            quantity=zp.fields.REFLECTANCE,
            residual_threshold=RESIDUAL_THRESHOLD,
        ),
        "validation/radiance_against_reference": zp.validation.overlay(
            spectrum,
            reference,
            quantity=zp.fields.RADIANCE,
            reference_label="Reference implementation",
            current_label="Zig implementation",
        ),
        "validation/validation_metrics": zp.validation.metrics_bar(spectrum, reference),
        "atmosphere/total_optical_depth_heatmap": zp.atmosphere.optical_depth_heatmap(
            bundle["budget"],
            quantity=zp.fields.TOTAL_OPTICAL_DEPTH,
            markers_nm=MARKERS_NM,
        ),
        "atmosphere/optical_depth_component_stack": zp.atmosphere.component_stack(bundle["budget"]),
        "cia/cia_share_spectrum": zp.cia.share_spectrum(bundle["cia"]),
        "instrument_response/isrf_" + wavelength_slug(760.76): zp.instrument_response.isrf(
            bundle["response"],
            nominal_wavelength_nm=760.76,
        ),
    }
    for wavelength_nm in selected_wavelengths:
        suffix = wavelength_slug(wavelength_nm)
        charts[f"atmosphere/total_optical_depth_profile_{suffix}"] = (
            zp.atmosphere.optical_depth_profile(
                bundle["budget"],
                quantity=zp.fields.TOTAL_OPTICAL_DEPTH,
                wavelengths_nm=(wavelength_nm,),
            )
        )
        charts[f"atmosphere/single_scatter_albedo_profile_{suffix}"] = (
            zp.atmosphere.single_scatter_albedo_profile(
                bundle["budget"],
                wavelengths_nm=(wavelength_nm,),
            )
        )
        charts[f"cia/cia_share_profile_{suffix}"] = zp.cia.share_profile(
            bundle["cia"],
            wavelength_nm=wavelength_nm,
        )
        charts[f"cia/cia_optical_depth_profile_{suffix}"] = zp.cia.optical_depth_profile(
            bundle["cia"],
            wavelength_nm=wavelength_nm,
        )
        charts[f"cia/cia_cross_section_temperature_{suffix}"] = zp.cia.cross_section_temperature(
            bundle["cia"],
            wavelength_nm=wavelength_nm,
        )
    return charts

## Execute and Interpret

The cell below displays each chart inline and returns a compact summary of the generated chart set.

In [ ]:
from IPython.display import Markdown, display

library_path = (
    REPO_ROOT
    / "zig-out"
    / "lib"
    / ("libzdisamar_c.dylib" if sys.platform == "darwin" else "libzdisamar_c.so")
)
if not library_path.exists():
    raise FileNotFoundError(f"{library_path} does not exist; build the native shared library first")
zd, zp = import_runtime()
zp.use_theme("validation")
bundle = collect_core_outputs(zd, zp, str(library_path), DEFAULT_REFERENCE)
charts = build_plots(zp, bundle)
for name, chart in charts.items():
    display(Markdown(f"### {name}"))
    display(chart)
summary = {
    "chart_count": len(charts),
    "sample_count": int(len(bundle["spectrum"])),
    "selected_wavelengths_nm": [float(value) for value in bundle["selected_wavelengths"]],
    "metrics": bundle["metrics"],
}
summary